# Team API Key Packet Generator

This notebook loads primary API keys from `keys.csv`, loads team rosters from `teams.csv`, assigns keys to students in order, and writes a PDF named `team_api_keys.pdf` in the same directory.

In [5]:
from pathlib import Path
import csv
import subprocess
import sys

try:
    from reportlab.lib.pagesizes import letter
    from reportlab.pdfgen import canvas
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "reportlab"])
    from reportlab.lib.pagesizes import letter
    from reportlab.pdfgen import canvas

BASE_DIR = Path.cwd()
KEYS_PATH = BASE_DIR / "keys.csv"
TEAMS_PATH = BASE_DIR / "teams.csv"
OUTPUT_PATH = BASE_DIR / "team_api_keys.pdf"

print(f"Working directory: {BASE_DIR}")
print(f"PDF output: {OUTPUT_PATH}")

Working directory: /workspaces/comp423-25s.github.io/scripts/keys
PDF output: /workspaces/comp423-25s.github.io/scripts/keys/team_api_keys.pdf


In [6]:
def load_primary_keys(keys_path: Path) -> list[str]:
    with keys_path.open(newline="", encoding="utf-8") as handle:
        rows = csv.DictReader(handle)
        return [row["Primary Key"].strip() for row in rows if row.get("Primary Key", "").strip()]


def load_teams(teams_path: Path) -> dict[str, list[tuple[str, str]]]:
    teams: dict[str, list[tuple[str, str]]] = {}
    with teams_path.open(newline="", encoding="utf-8") as handle:
        rows = csv.DictReader(handle)
        for row in rows:
            team_name = row["Team/Table"].strip()
            members: list[tuple[str, str]] = []
            for index in range(1, 5):
                name = (row.get(f"Member {index} Name") or "").strip()
                pid = (row.get(f"Member {index} PID") or "").strip()
                if name and pid:
                    members.append((name, pid))
            teams[team_name] = members
    return teams


primary_keys = load_primary_keys(KEYS_PATH)
teams = load_teams(TEAMS_PATH)
student_count = sum(len(members) for members in teams.values())

print(f"Loaded {len(primary_keys)} primary keys.")
print(f"Loaded {len(teams)} teams and {student_count} students.")
print(next(iter(teams.items())))

Loaded 200 primary keys.
Loaded 51 teams and 199 students.
('A1', [('Daniel Wang', '730677774'), ('Nicholas Do', '730825090'), ('Olivia Liu', '730667914'), ('Yahan Yang', '730649168')])


In [7]:
def assign_keys_to_teams(
    teams: dict[str, list[tuple[str, str]]],
    primary_keys: list[str],
) -> dict[str, list[tuple[str, str, str]]]:
    assigned: dict[str, list[tuple[str, str, str]]] = {}
    key_index = 0

    for team_name, members in teams.items():
        assigned_members: list[tuple[str, str, str]] = []
        for name, pid in members:
            if key_index >= len(primary_keys):
                raise ValueError("Not enough primary keys to assign every student.")
            assigned_members.append((name, pid, primary_keys[key_index]))
            key_index += 1
        assigned[team_name] = assigned_members

    return assigned


assigned_teams = assign_keys_to_teams(teams, primary_keys)
assigned_count = sum(len(members) for members in assigned_teams.values())
print(f"Assigned keys to {assigned_count} students.")
print(next(iter(assigned_teams.items())))

Assigned keys to 199 students.
('A1', [('Daniel Wang', '730677774', '4358e5f613404e8ab05a5cce655f93db'), ('Nicholas Do', '730825090', 'f4a61aedbb73409b87f004e26edc536e'), ('Olivia Liu', '730667914', '35f71b20e4fb43bcb960f8a200d4d54f'), ('Yahan Yang', '730649168', 'ed1c1b8cff734390af220d2c9dc4c446')])


In [8]:
def draw_labeled_value(pdf: canvas.Canvas, x: float, y: float, label: str, value: str) -> None:
    pdf.setFont("Helvetica", 12)
    pdf.drawString(x, y, label)
    label_width = pdf.stringWidth(label, "Helvetica", 12)
    pdf.setFont("Helvetica-Bold", 12)
    pdf.drawString(x + label_width + 4, y, value)


def generate_team_pdf(assigned_teams: dict[str, list[tuple[str, str, str]]], output_path: Path) -> Path:
    pdf = canvas.Canvas(str(output_path), pagesize=letter)
    page_width, page_height = letter
    left_margin = 54
    right_margin = 54
    top_margin = 54
    bottom_margin = 54
    heading_y = page_height - top_margin
    section_top = heading_y - 36
    section_height = (page_height - top_margin - bottom_margin) * 0.20
    usable_width = page_width - left_margin - right_margin

    for team_name, members in assigned_teams.items():
        pdf.setTitle("Team API Keys")
        pdf.setFont("Helvetica-Bold", 22)
        pdf.drawString(left_margin, heading_y, f"TEAM {team_name}")

        for member_index, (name, pid, key) in enumerate(members):
            top_y = section_top - (member_index * section_height)
            text_y = top_y - 20

            draw_labeled_value(pdf, left_margin, text_y, "Name:", name)
            draw_labeled_value(pdf, left_margin, text_y - 24, "PID:", pid)
            draw_labeled_value(pdf, left_margin, text_y - 48, "API Key:", key)

            if member_index < len(members) - 1:
                separator_y = top_y - section_height + 8
                pdf.setDash(6, 4)
                pdf.line(left_margin, separator_y, left_margin + usable_width, separator_y)
                pdf.setDash()

        pdf.showPage()

    pdf.save()
    return output_path


pdf_path = generate_team_pdf(assigned_teams, OUTPUT_PATH)
print(f"Saved PDF to {pdf_path}")

Saved PDF to /workspaces/comp423-25s.github.io/scripts/keys/team_api_keys.pdf


In [9]:
CSV_OUTPUT_PATH = BASE_DIR / "team_api_key_assignments.csv"

rows = []
for team_name, members in assigned_teams.items():
    for name, pid, primary_key in members:
        rows.append(
            {
                "Team": team_name,
                "Name": name,
                "PID": pid,
                "Primary Key": primary_key,
            }
        )

with CSV_OUTPUT_PATH.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["Team", "Name", "PID", "Primary Key"],
    )
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved {len(rows)} student assignments to {CSV_OUTPUT_PATH}")
CSV_OUTPUT_PATH

Saved 199 student assignments to /workspaces/comp423-25s.github.io/scripts/keys/team_api_key_assignments.csv


PosixPath('/workspaces/comp423-25s.github.io/scripts/keys/team_api_key_assignments.csv')